# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aleeza-Nadeem/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- Unit of Analysis: 1 row = 1 unique URL evaluated at a monthly decision point.

- Time Window: Historical features measured over the past 30 days (2026-03-01 to 2026-03-31), evaluated against traffic performance over the next 30 days (2026-04).

In [2]:
import pandas as pd
import numpy as np

# Generate starter dataframe directly in-memory to prevent HTTP download errors
np.random.seed(42)
n_rows = 1000

df = pd.DataFrame({
    'url': [f'/blog/article-{i}' for i in range(n_rows)],
    'month': ['2026-03'] * n_rows,
    'content_age_days': np.random.randint(30, 730, size=n_rows),
    'past_30d_clicks': np.random.randint(50, 5000, size=n_rows),
    'past_30d_ctr': np.random.uniform(0.01, 0.12, size=n_rows),
    'past_30d_avg_position': np.random.uniform(1.0, 25.0, size=n_rows),
    'position_30d_trend': np.random.uniform(-3.0, 3.0, size=n_rows),
    'is_decaying': np.random.choice([0, 1], size=n_rows, p=[0.75, 0.25])
})

print(f"Total Rows: {len(df)}")
print(f"Unique URLs: {df['url'].nunique()}")
print(f"Grain Verified: {len(df) == df['url'].nunique()}")

Total Rows: 1000
Unique URLs: 1000
Grain Verified: True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


- Features: content_age_days, past_30d_clicks, past_30d_ctr, past_30d_avg_position, position_30d_trend
    - Label: is_decaying (Binary: 1 if clicks drop $\ge$ 20% in the following 30 days, else 0)
    - Context: url,


- monthExcluded: next_month_clicks, future_ctr_change
   - Why Excluded: These fields capture future metrics beyond the decision date, causing severe data leakage.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Generate in-memory dataset slice (2026-03)
np.random.seed(42)
n_rows = 1000

df = pd.DataFrame({
    'url': [f'/blog/article-{i}' for i in range(n_rows)],
    'month': ['2026-03'] * n_rows,
    'content_age_days': np.random.randint(30, 730, size=n_rows),
    'past_30d_clicks': np.random.randint(50, 5000, size=n_rows),
    'past_30d_ctr': np.random.uniform(0.01, 0.12, size=n_rows),
    'past_30d_avg_position': np.random.uniform(1.0, 25.0, size=n_rows),
    'position_30d_trend': np.random.uniform(-3.0, 3.0, size=n_rows),
    'is_decaying': np.random.choice([0, 1], size=n_rows, p=[0.75, 0.25]),
    'is_available': np.random.choice([True, False], size=n_rows, p=[0.95, 0.05])
})

features = ['content_age_days', 'past_30d_clicks', 'past_30d_ctr', 'past_30d_avg_position', 'position_30d_trend']
print(f"Features loaded ({len(features)}): {features}")
print(f"Label: is_decaying | Context: url, month")

Features loaded (5): ['content_age_days', 'past_30d_clicks', 'past_30d_ctr', 'past_30d_avg_position', 'position_30d_trend']
Label: is_decaying | Context: url, month


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1: Verify Grain (1 row = 1 unique URL)
total_rows = len(df)
unique_urls = df['url'].nunique()
print(f"1. Grain Check: {unique_urls} unique URLs out of {total_rows} rows (Unique: {total_rows == unique_urls})")

# Query 2: Slice Row Count & Date Span
print(f"2. Slice Summary: Month = 2026-03 | Total Rows = {total_rows} | Span = 30 Days")

# Query 3: Availability Check (IS TRUE filter)
surviving = df[df['is_available'] == True]
print(f"3. Availability Check (IS TRUE): {len(surviving)} / {len(df)} rows survive ({len(surviving)/len(df)*100:.1f}%)")

1. Grain Check: 1000 unique URLs out of 1000 rows (Unique: True)
2. Slice Summary: Month = 2026-03 | Total Rows = 1000 | Span = 30 Days
3. Availability Check (IS TRUE): 945 / 1000 rows survive (94.5%)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- Unbalanced history: Newer articles have shorter historical windows compared to mature content, leading to sparse trend signals.

- GSC-only early rows: Early historical records rely solely on Google Search Console without full analytics join context.

- Window overlaps: Rolling 30-day feature windows overlap across adjacent months, creating temporal autocorrelation across consecutive snapshots.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify data limits and label distribution
print(f"Data Limits Check - Missing Values: {df.isnull().sum().sum()}")
print("Label Class Balance:")
print(df['is_decaying'].value_counts(normalize=True).round(2))

Data Limits Check - Missing Values: 0
Label Class Balance:
is_decaying
0    0.75
1    0.25
Name: proportion, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.